# Game Intelligence AI — 02 Model Benchmark (Automatic GPU)

This notebook detects the GPU assigned by Colab and selects conservative inference settings automatically. It does not mount Google Drive.

**Storage:** GitHub for source/notebooks; Hugging Face Hub for model/data artifacts within current quotas and license terms; `/content` for temporary runtime files. `/content` is deleted when the runtime ends. Never commit tokens, large datasets, model weights or checkpoints to GitHub.

Run from top to bottom. Select Runtime → Change runtime type → GPU if available. Do not manually replace PyTorch/CUDA.

In [ ]:
import os, sys, json, time, gc, shutil, subprocess, platform
from pathlib import Path
from datetime import datetime, timezone
PROJECT_ROOT=Path("/content/GameIntelligenceAI")
for d in ["benchmarks","logs","models/cache","tmp"]:
    (PROJECT_ROOT/d).mkdir(parents=True,exist_ok=True)
print("Python:",sys.version)
print("Platform:",platform.platform())
print("Project:",PROJECT_ROOT)

## 1. Detect assigned hardware

In [ ]:
import torch
GPU_AVAILABLE=torch.cuda.is_available()
GPU_NAME=torch.cuda.get_device_name(0) if GPU_AVAILABLE else None
GPU_VRAM_GB=(torch.cuda.get_device_properties(0).total_memory/1024**3) if GPU_AVAILABLE else 0
CUDA_VERSION=torch.version.cuda
total,used,free=shutil.disk_usage("/content")
print("GPU:",GPU_NAME)
print("VRAM GB:",round(GPU_VRAM_GB,2))
print("CUDA:",CUDA_VERSION,"| PyTorch:",torch.__version__)
print("Free /content GB:",round(free/1024**3,2))
p=subprocess.run(["bash","-lc","nvidia-smi"],capture_output=True,text=True)
print(p.stdout if p.returncode==0 else p.stderr)
if not GPU_AVAILABLE:
    PROFILE={"mode":"CPU","quant4":False,"max_new_tokens":96,"max_input_tokens":1024}
elif GPU_VRAM_GB<8:
    PROFILE={"mode":"LOW_VRAM","quant4":True,"max_new_tokens":128,"max_input_tokens":2048}
elif GPU_VRAM_GB<12:
    PROFILE={"mode":"MID_VRAM","quant4":True,"max_new_tokens":160,"max_input_tokens":3072}
elif GPU_VRAM_GB<20:
    PROFILE={"mode":"T4_CLASS_16GB","quant4":True,"max_new_tokens":192,"max_input_tokens":3072}
elif GPU_VRAM_GB<32:
    PROFILE={"mode":"20-31GB","quant4":True,"max_new_tokens":256,"max_input_tokens":4096}
else:
    PROFILE={"mode":"HIGH_VRAM","quant4":True,"max_new_tokens":384,"max_input_tokens":6144}
print("Automatic profile:",PROFILE)

## 2. Check inference libraries

In [ ]:
import importlib.metadata as md
for p in ["transformers","accelerate","bitsandbytes","huggingface-hub","safetensors"]:
    try: print(p,md.version(p))
    except Exception as e: print(p,"MISSING",e)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import model_info
print("Imports OK")

## 3. Candidate models

These are benchmark candidates, not a claim of global ranking. Confirm each official model card, current revision, license and commercial-use conditions before adopting it. If a candidate is unavailable or incompatible, the notebook records the error and continues.

In [ ]:
CANDIDATES=[
 {"id":"Qwen/Qwen2.5-Coder-3B-Instruct","label":"Qwen2.5-Coder 3B","class":"small"},
 {"id":"Qwen/Qwen2.5-Coder-7B-Instruct","label":"Qwen2.5-Coder 7B","class":"medium"},
 {"id":"Qwen/Qwen3-4B","label":"Qwen3 4B","class":"small"},
 {"id":"Qwen/Qwen3-8B","label":"Qwen3 8B","class":"medium"},
]
def allowed(c):
    if not GPU_AVAILABLE: return False
    if GPU_VRAM_GB<12: return c["class"]=="small"
    return True
SELECTED=[c for c in CANDIDATES if allowed(c)]
print("Selected:",[c["id"] for c in SELECTED])
print("Skipped:",[c["id"] for c in CANDIDATES if c not in SELECTED])

## 4. Optional Hugging Face authentication

For public model repositories, a token is usually unnecessary. For gated/private models or uploads, add a Colab Secret named `HF_TOKEN`. Never paste a token into a cell or print it.

In [ ]:
HF_TOKEN=None
try:
    from google.colab import userdata
    HF_TOKEN=userdata.get("HF_TOKEN")
except Exception:
    pass
print("HF token configured:",bool(HF_TOKEN))

## 5. Game-development benchmark prompts

A quick smoke benchmark with two prompts per model by default. This is not a complete or definitive quality evaluation.

In [ ]:
TASKS=[
 {"id":"web_platformer","prompt":"Write a self-contained JavaScript updatePlayer(player,input,dt) for a 2D platformer. Include delta-time movement, gravity, jump only when grounded. Return code only.","checks":["function","gravity","ground"]},
 {"id":"godot_pickup","prompt":"Write Godot 4 GDScript for a coin Area2D that detects a player, prevents double collection, emits a signal and calls queue_free. Return code only.","checks":["Area2D","signal","queue_free"]},
 {"id":"unity_health","prompt":"Write a minimal Unity C# MonoBehaviour health component with max health, damage, clamp at zero and a death event. Return code only.","checks":["MonoBehaviour","Mathf","event"]},
 {"id":"debug_js","prompt":"Explain and fix: for (var i=0;i<3;i++){setTimeout(()=>console.log(i),0)}. Provide corrected code and reason.","checks":["let","0","1","2"]},
 {"id":"level_design","prompt":"Design an original beginner 2D puzzle-platformer level. Include objective, mechanics, layout, hazards, accessibility and measurable completion condition.","checks":["objective","mechanic","hazard","completion"]}
]
TASK_LIMIT=2

## 6. Model loading and GPU-safe cleanup

The notebook uses 4-bit quantization on supported CUDA GPUs. If loading fails, it records the error; it does not silently try a large full-precision model on a small GPU.

In [ ]:
import torch
def cleanup(model=None,tokenizer=None):
    del model,tokenizer
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def load_model(model_id):
    tok=AutoTokenizer.from_pretrained(model_id,token=HF_TOKEN)
    kw={"token":HF_TOKEN}
    if GPU_AVAILABLE:
        kw.update({
          "device_map":"auto",
          "torch_dtype":torch.float16,
          "quantization_config":BitsAndBytesConfig(
             load_in_4bit=True,bnb_4bit_quant_type="nf4",
             bnb_4bit_compute_dtype=torch.float16,
             bnb_4bit_use_double_quant=True)
        })
    else:
        kw["torch_dtype"]=torch.float32
    model=AutoModelForCausalLM.from_pretrained(model_id,**kw)
    if tok.pad_token_id is None: tok.pad_token=tok.eos_token
    return tok,model

## 7. Run benchmark sequentially

In [ ]:
RESULTS=[]
def format_prompt(tok,text):
    msgs=[{"role":"user","content":text}]
    if getattr(tok,"chat_template",None):
        return tok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
    return text

for cand in SELECTED:
    rec={"model_id":cand["id"],"label":cand["label"],"loaded":False,"tasks":[],"error":None}
    tok=model=None
    try:
        if GPU_AVAILABLE: torch.cuda.reset_peak_memory_stats()
        t=time.time()
        tok,model=load_model(cand["id"])
        rec["load_seconds"]=round(time.time()-t,2)
        rec["loaded"]=True
        for task in TASKS[:TASK_LIMIT]:
            prompt=format_prompt(tok,task["prompt"])
            inp=tok(prompt,return_tensors="pt",truncation=True,max_length=PROFILE["max_input_tokens"])
            inp={k:v.to(model.device) for k,v in inp.items()}
            start=time.time()
            with torch.inference_mode():
                out=model.generate(**inp,max_new_tokens=PROFILE["max_new_tokens"],
                    do_sample=False,pad_token_id=tok.pad_token_id)
            elapsed=time.time()-start
            new=out[0,inp["input_ids"].shape[1]:]
            answer=tok.decode(new,skip_special_tokens=True)
            score=sum(x.lower() in answer.lower() for x in task["checks"])
            item={"task":task["id"],"seconds":round(elapsed,2),
                  "tokens":int(new.numel()),
                  "tokens_per_second":round(new.numel()/max(elapsed,0.001),2),
                  "simple_checks":f"{score}/{len(task['checks'])}",
                  "output":answer[:3500]}
            rec["tasks"].append(item)
            print(cand["label"],task["id"],item["tokens_per_second"],"tok/s",item["simple_checks"])
        if GPU_AVAILABLE: rec["peak_vram_gb"]=round(torch.cuda.max_memory_allocated()/1024**3,2)
    except Exception as e:
        rec["error"]=f"{type(e).__name__}: {str(e)[:1000]}"
        print("FAILED",cand["id"],rec["error"])
    finally:
        cleanup(model,tok)
    RESULTS.append(rec)
    (PROJECT_ROOT/"benchmarks"/"partial_results.json").write_text(json.dumps(RESULTS,indent=2,ensure_ascii=False))

## 8. Summarize measured results

Simple string checks are not semantic evaluation. Review the outputs manually and later add tests that execute generated code in an isolated environment.

In [ ]:
SUMMARY=[]
for r in RESULTS:
    vals=[]
    speeds=[]
    for t in r.get("tasks",[]):
        try:
            a,b=t["simple_checks"].split("/")
            vals.append(int(a)/int(b))
        except Exception: pass
        speeds.append(t.get("tokens_per_second",0))
    SUMMARY.append({
      "model":r["model_id"],"loaded":r["loaded"],
      "tasks":len(r["tasks"]),
      "mean_simple_check":round(sum(vals)/len(vals),3) if vals else None,
      "mean_tokens_per_second":round(sum(speeds)/len(speeds),2) if speeds else None,
      "peak_vram_gb":r.get("peak_vram_gb"),"error":r.get("error")
    })
print(json.dumps(SUMMARY,indent=2,ensure_ascii=False))

## 9. Save and download report

In [ ]:
REPORT={
 "project":"Game Intelligence AI",
 "created_utc":datetime.now(timezone.utc).isoformat(),
 "hardware":{"gpu_available":GPU_AVAILABLE,"gpu_name":GPU_NAME,
             "gpu_vram_gb":round(GPU_VRAM_GB,2),"cuda":CUDA_VERSION,
             "torch":torch.__version__,"profile":PROFILE},
 "free_disk_gb_at_start":round(free/1024**3,2),
 "results":RESULTS,"summary":SUMMARY,
 "limitations":["Small prompt set, not comprehensive.",
 "String checks are heuristic, not semantic evaluation.",
 "Inference success does not prove QLoRA training will fit.",
 "Check model cards, licenses and revision before commercial use."]
}
REPORT_PATH=PROJECT_ROOT/"benchmarks"/"model_benchmark_report.json"
REPORT_PATH.write_text(json.dumps(REPORT,indent=2,ensure_ascii=False))
print("Report:",REPORT_PATH)
try:
    from google.colab import files
    files.download(str(REPORT_PATH))
except Exception as e:
    print("Download unavailable:",e)

## 10. What to do with the result

Do not fine-tune yet. Send the downloaded `model_benchmark_report.json` here. We will use the actual results to select a base model and design a separate training notebook.

Before training, verify the chosen model's official license, dataset provenance, memory requirements, and code execution tests. Keep source/notebooks in GitHub; keep large datasets/adapters/checkpoints in a persistent repository such as Hugging Face Hub only if current storage, bandwidth and license terms fit. `/content` is temporary; no Google Drive is used.

In [ ]:
print("Benchmark complete.")
print("GPU profile:",PROFILE["mode"])
print("Models attempted:",len(RESULTS))
print("Successful loads:",sum(x["loaded"] for x in RESULTS))